<a href="https://colab.research.google.com/github/MalakMostafa-16003651/nyc-housing-transportation-analysis/blob/feature%2Fdata-integration/dataEngP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd

df_housing_clean = pd.read_csv("df_housing_clean.csv")
df_transport_clean = pd.read_csv("df_transport_clean.csv")

for df in [df_housing_clean, df_transport_clean]:
    df["zip_code"] = pd.to_numeric(df["zip_code"], errors="coerce").astype("Int64").astype("string").str.zfill(5)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

print("Housing:", df_housing_clean.shape)
print("Transportation:", df_transport_clean.shape)

Housing: (9551, 42)
Transportation: (8072, 26)


In [12]:
keys = ["zip_code", "year"]

# Exclude rows missing integration keys
housing = df_housing_clean.dropna(subset=keys).copy()
transport = df_transport_clean.dropna(subset=keys).copy()

# Housing aggregation
df_housing_agg = housing.groupby(keys, as_index=False).agg(
    housing_borough=("borough", lambda x: " / ".join(sorted(x.dropna().unique()))),
    housing_building_count=("building_id", "nunique"),
    housing_project_count=("project_id", "nunique"),
    affordable_units=("all_counted_units", "sum"),
    total_housing_units=("total_units", "sum")
)

# Transportation aggregation
df_transport_agg = transport.groupby(keys, as_index=False).agg(
    transport_borough=("borough", lambda x: " / ".join(sorted(x.dropna().unique()))),
    serious_crash_count=("open_data_crash_id", "nunique"),
    severe_injuries=("total_severe_injuries", "sum"),
    non_severe_injuries=("total_non_severe_injuries", "sum"),
    fatalities=("total_fatalities", "sum")
)

print("Housing aggregated:", df_housing_agg.shape)
print("Transport aggregated:", df_transport_agg.shape)
print("Housing duplicate keys:", df_housing_agg.duplicated(keys).sum())
print("Transport duplicate keys:", df_transport_agg.duplicated(keys).sum())

Housing aggregated: (1155, 7)
Transport aggregated: (829, 7)
Housing duplicate keys: 0
Transport duplicate keys: 0


In [13]:
keys = ["zip_code", "year"]

merge_audit = df_housing_agg.merge(
    df_transport_agg,
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

print("Merge results:")
print(merge_audit["_merge"].value_counts())

df_integrated = merge_audit[
    merge_audit["_merge"] == "both"
].drop(columns="_merge").copy()

print("\nFinal merged shape:", df_integrated.shape)

print("Borough disagreements:", (
    df_integrated["housing_borough"] !=
    df_integrated["transport_borough"]
).sum())

Merge results:
_merge
left_only     671
both          484
right_only    345
Name: count, dtype: int64

Final merged shape: (484, 12)
Borough disagreements: 6


In [14]:
df_integrated.loc[
    df_integrated["housing_borough"] != df_integrated["transport_borough"],
    ["zip_code", "year", "housing_borough", "transport_borough"]
]

,zip_code,year,housing_borough,transport_borough
606,10463,2024,Bronx / Manhattan,Bronx
607,10463,2025,Bronx / Manhattan,Bronx
1131,11237,2023,Brooklyn,Brooklyn / Queens
1134,11237,2026,Brooklyn,Brooklyn / Queens
1313,11385,2022,Queens,Brooklyn / Queens
1314,11385,2023,Queens,Brooklyn / Queens


In [15]:
# Choose the most frequent housing borough for each ZIP code
borough_map = housing.groupby("zip_code")["borough"].agg(
    lambda x: x.mode().iloc[0]
)

df_integrated["borough"] = df_integrated["zip_code"].map(borough_map)

print(df_integrated["borough"].value_counts())
print("Missing boroughs:", df_integrated["borough"].isna().sum())

borough
Brooklyn         153
Manhattan        111
Queens           106
Bronx            102
Staten Island     12
Name: count, dtype: int64
Missing boroughs: 0


In [16]:
metrics = [
    "housing_building_count", "housing_project_count",
    "affordable_units", "total_housing_units",
    "serious_crash_count", "severe_injuries",
    "non_severe_injuries", "fatalities"
]

print("Missing values:")
print(df_integrated.isna().sum())

print("\nDuplicate ZIP-year rows:",
      df_integrated.duplicated(["zip_code", "year"]).sum())

print("Negative numeric values:",
      (df_integrated[metrics] < 0).sum().sum())

Missing values:
zip_code                  0
year                      0
housing_borough           0
housing_building_count    0
housing_project_count     0
affordable_units          0
total_housing_units       0
transport_borough         0
serious_crash_count       0
severe_injuries           0
non_severe_injuries       0
fatalities                0
borough                   0
dtype: int64

Duplicate ZIP-year rows: 0
Negative numeric values: 0


In [17]:
df_integrated = (
    df_integrated
    .drop(columns=["housing_borough", "transport_borough"])
    .sort_values(["year", "zip_code"])
    .reset_index(drop=True)
)

df_integrated["year"] = df_integrated["year"].astype(int)

df_integrated.to_csv("df_integrated.csv", index=False)

print("Final dataset:", df_integrated.shape)
print("Saved successfully!")

Final dataset: (484, 11)
Saved successfully!


### Data Integration and Post-Integration Cleaning

**1. Integration approach**

We aggregated both datasets by `zip_code` and `year` before merging. This ensures each ZIP code has only one record per year and prevents duplicate matches.

**2. Missing ZIP codes**

We excluded 1,958 housing records with missing ZIP codes because they could not be reliably matched. Although these represented 20.5% of housing rows, they accounted for only 1.04% of affordable housing units.

**3. Merging**

We used an inner join on `zip_code` and `year` to keep records with information from both datasets.

- 484 matched records
- 671 housing-only records
- 345 transportation-only records

**4. Post-integration cleaning**

We checked the merged dataset for missing values, duplicate ZIP-year combinations, negative values, and borough inconsistencies. Six borough disagreements were found. We assigned each ZIP code its most frequent housing borough for consistent filtering and removed the redundant borough columns.

**5. Final validation**

The final dataset contains 484 rows and 11 columns, with no missing values, duplicate integration keys, or negative numeric values.

**6. Limitations**

- Records without matching ZIP-year combinations were excluded.
- Some ZIP codes span multiple boroughs.
- The datasets do not cover identical observation periods for every year.
- The analysis identifies associations, not causal relationships between housing development and road safety.

In [18]:
import plotly.express as px

rq1 = df_integrated[df_integrated["year"].between(2023, 2025)].copy()

housing_limit = rq1["affordable_units"].quantile(0.75)
crash_limit = rq1["serious_crash_count"].quantile(0.75)

rq1["high_both"] = (
    (rq1["affordable_units"] >= housing_limit) &
    (rq1["serious_crash_count"] >= crash_limit)
)

rq1["category"] = rq1["high_both"].map({
    True: "High in both",
    False: "Other"
})

fig = px.scatter(
    rq1,
    x="affordable_units",
    y="serious_crash_count",
    color="category",
    hover_data=["zip_code", "year", "borough"],
    title="Affordable Housing vs Serious Crashes (2023–2025)",
    labels={
        "affordable_units": "Affordable Housing Units",
        "serious_crash_count": "Serious Crash Count",
        "category": "Category"
    }
)

fig.add_vline(x=housing_limit, line_dash="dash")
fig.add_hline(y=crash_limit, line_dash="dash")
fig.show()

print("ZIP-year records with high housing and high crashes:")
display(
    rq1.loc[rq1["high_both"],
    ["zip_code", "year", "borough", "affordable_units", "serious_crash_count"]]
    .sort_values("serious_crash_count", ascending=False)
    .head(10)
)

ZIP-year records with high housing and high crashes:


,zip_code,year,borough,affordable_units,serious_crash_count
365,11212,2025,Brooklyn,1090.0,43.0
335,10456,2025,Bronx,923.0,36.0
146,11207,2023,Brooklyn,702.0,36.0
131,10468,2023,Bronx,516.0,32.0
121,10456,2023,Bronx,712.0,30.0
361,11208,2025,Brooklyn,612.0,30.0
228,10457,2024,Bronx,1105.0,28.0
236,10467,2024,Bronx,753.0,27.0
332,10453,2025,Bronx,522.0,27.0
373,11220,2025,Brooklyn,811.0,27.0


**Finding:** Several ZIP codes in Brooklyn and the Bronx showed both high affordable housing production and high serious-crash counts during 2023–2025. For example, ZIP code 11212 (Brooklyn) recorded 1,090 affordable units and 43 serious crashes in 2025. This suggests overlapping housing and road-safety concerns, but does not establish causation.

In [19]:
rq2 = (
    df_integrated[df_integrated["year"].between(2023, 2025)]
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        affordable_units=("affordable_units", "sum"),
        serious_crash_count=("serious_crash_count", "sum"),
        years_observed=("year", "nunique")
    )
)

rq2 = rq2[
    (rq2["years_observed"] == 3) &
    (rq2["affordable_units"] >= 100)
].copy()

rq2["crashes_per_100_units"] = (
    rq2["serious_crash_count"] / rq2["affordable_units"] * 100
)

top10 = rq2.nlargest(10, "crashes_per_100_units")

fig = px.bar(
    top10.sort_values("crashes_per_100_units"),
    x="crashes_per_100_units",
    y="zip_code",
    color="borough",
    orientation="h",
    hover_data=["affordable_units", "serious_crash_count"],
    title="Top 10 ZIP Codes: Serious Crashes Relative to Affordable Units",
    labels={
        "crashes_per_100_units": "Crashes per 100 Affordable Units",
        "zip_code": "ZIP Code"
    }
)

fig.show()

display(top10[
    ["zip_code", "borough", "affordable_units",
     "serious_crash_count", "crashes_per_100_units"]
].round(2))

,zip_code,borough,affordable_units,serious_crash_count,crashes_per_100_units
99,11234,Brooklyn,157.0,103.0,65.61
119,11377,Queens,103.0,67.0,65.05
54,10466,Bronx,135.0,62.0,45.93
127,11432,Queens,149.0,64.0,42.95
90,11223,Brooklyn,139.0,55.0,39.57
73,11206,Brooklyn,396.0,89.0,22.47
23,10032,Manhattan,195.0,42.0,21.54
102,11237,Brooklyn,198.0,39.0,19.70
52,10462,Bronx,274.0,47.0,17.15
105,11249,Brooklyn,167.0,27.0,16.17


**Finding:** ZIP codes 11234 (Brooklyn) and 11377 (Queens) had the highest serious-crash counts relative to affordable housing production during 2023–2025. This comparison helps identify areas worth further investigation, but it does not represent actual crash risk because housing units are not a measure of traffic exposure.

## Reproduction and interpretation audit
The committed inputs reproduce 484 matched ZIP-year rows. The outer join contains 671 housing-only and 345 transport-only groups; these are excluded rather than treated as zero. Six matched rows have differing borough summaries; the existing policy assigns each ZIP its most frequent housing borough, which is an analytical convention rather than a geographic boundary correction.
Housing has 29 missing all_counted_units values. No ZIP-year group is entirely missing, but sum skips unknown rows: affordable-unit totals are observed sums and may undercount. Do not claim all source missing values were resolved merely because the aggregated output has no nulls. Housing includes preservation and new construction. Transportation dates in the cleaned snapshot span 2022-10-01 to 2026-06-30.
Reproduce and compare the transformation without rewriting the committed CSV: python -m unittest discover -s tests -v. The integration regression test executes the notebook transformation and compares all rows and columns after consistent ZIP typing.